# RelCheck Fast Test — Pipeline Improvements

Standalone notebook for quick iteration on pipeline fixes using ~20 images.

**Three fixes being tested:**
1. **VQA-Guided Correction** — LLaVA describes what the image actually shows before the corrector edits
2. **Tighter Thresholds** — Hallucination threshold lowered from `<0.50` to `<0.40` to reduce false positives
3. **Correction Validation** — Re-verify corrected triples through VQA; revert if correction fails

**Evaluation:** Mention-filtered LLM-judge R-POPE on the fast subset.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 1 — Setup & Install
# ═══════════════════════════════════════════════════════════════
!pip install -q together transformers bitsandbytes accelerate spacy Pillow torch torchvision
!python -m spacy download en_core_web_sm -q

import os
os.environ['TOGETHER_API_KEY'] = ''  # ← PASTE YOUR KEY HERE

# Clone / update repo
REPO_DIR = '/content/RelCheck'
if os.path.exists(f'{REPO_DIR}/.git'):
    !cd {REPO_DIR} && git pull
else:
    import shutil
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    !git clone https://github.com/siddhipatil/RelCheck.git {REPO_DIR}

import sys
sys.path.insert(0, f'{REPO_DIR}/relcheck')

print('✅ Setup complete')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 2 — Load Models (BLIP-2 + LLaVA)
# ═══════════════════════════════════════════════════════════════
import torch
from transformers import (
    Blip2Processor, Blip2ForConditionalGeneration,
    LlavaForConditionalGeneration, AutoProcessor,
    BitsAndBytesConfig,
)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

# BLIP-2 (captioning)
print('Loading BLIP-2...')
blip2_processor = Blip2Processor.from_pretrained('Salesforce/blip2-flan-t5-xl')
blip2_model = Blip2ForConditionalGeneration.from_pretrained(
    'Salesforce/blip2-flan-t5-xl',
    torch_dtype=torch.float16,
).to(device)
blip2_model.eval()
print('✅ BLIP-2 loaded')

# LLaVA (verification)
print('Loading LLaVA-1.5-7B...')
bnb_config = BitsAndBytesConfig(load_in_8bit=True)
llava_processor = AutoProcessor.from_pretrained('llava-hf/llava-1.5-7b-hf')
llava_model = LlavaForConditionalGeneration.from_pretrained(
    'llava-hf/llava-1.5-7b-hf',
    quantization_config=bnb_config,
    device_map='auto',
)
llava_model.eval()
print('✅ LLaVA loaded')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — Prepare Fast Test Subset (~20 images)
# ═══════════════════════════════════════════════════════════════
import json
import pandas as pd
import random

EVAL_DIR = f'{REPO_DIR}/eval'

# Load R-Bench subset
with open(f'{EVAL_DIR}/rbench_subset.json') as f:
    eval_dataset = json.load(f)

# Load existing B1 results to get captions
df_b1 = pd.read_csv(f'{EVAL_DIR}/baseline_no_correction.csv')

# Get unique image IDs
all_image_ids = df_b1['image_id'].unique().tolist()

# Select 20 images: prioritize images that HAD corrections in previous run
# (they're more interesting for testing fixes)
df_rc_prev = pd.read_csv(f'{EVAL_DIR}/relcheck_results.csv')
corrected_ids = set(
    df_rc_prev[df_rc_prev['edit_rate'] > 0]['image_id'].unique()
)
uncorrected_ids = set(all_image_ids) - corrected_ids

# Take up to 15 corrected + 5 uncorrected for a balanced test
corrected_sample = random.sample(
    list(corrected_ids), min(15, len(corrected_ids))
)
uncorrected_sample = random.sample(
    list(uncorrected_ids), min(5, len(uncorrected_ids))
)
FAST_IDS = set(corrected_sample + uncorrected_sample)

print(f'Fast test subset: {len(FAST_IDS)} images')
print(f'  Corrected (prev run): {len(corrected_sample)}')
print(f'  Uncorrected:          {len(uncorrected_sample)}')

# Build image_id → caption map + image_id → path map
b1_cap_col = 'blip2_caption' if 'blip2_caption' in df_b1.columns else 'original_caption'
id_to_caption = {}
for img_id in FAST_IDS:
    rows = df_b1[df_b1['image_id'] == img_id]
    if len(rows) > 0:
        id_to_caption[img_id] = str(rows.iloc[0][b1_cap_col])

# Image paths — try R-Bench paths, fall back to Google Drive
id_to_path = {}
try:
    id_to_imgpath = {item['image_id']: item['image_path'] for item in eval_dataset}
except:
    id_to_imgpath = {}

# Try common locations
DRIVE_IMAGES_DIR = '/content/drive/MyDrive/CS298/eval/nocaps_images'
LOCAL_IMAGES_DIR = f'{REPO_DIR}/images'

for img_id in FAST_IDS:
    # Try R-Bench path
    p = id_to_imgpath.get(img_id, '')
    if p and os.path.exists(p):
        id_to_path[img_id] = p
        continue
    # Try Drive
    for ext in ['.jpg', '.jpeg', '.png', '.webp']:
        p = os.path.join(DRIVE_IMAGES_DIR, f'{img_id}{ext}')
        if os.path.exists(p):
            id_to_path[img_id] = p
            break

print(f'Images found: {len(id_to_path)}/{len(FAST_IDS)}')
print(f'Captions loaded: {len(id_to_caption)}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 4 — Run Improved RelCheck Pipeline
# ═══════════════════════════════════════════════════════════════
from relcheck_pipeline import RelCheckPipeline

pipeline = RelCheckPipeline(
    together_api_key=os.environ.get('TOGETHER_API_KEY'),
    use_llm_extractor=True,
    llava_model=llava_model,
    llava_processor=llava_processor,
    num_paraphrases=3,
)

results = []
for img_id in sorted(id_to_path.keys()):
    image_path = id_to_path[img_id]
    caption = id_to_caption.get(img_id)
    if not caption:
        continue
    
    print(f'\n{"="*60}')
    print(f'Image: {img_id}')
    print(f'{"="*60}')
    
    result = pipeline.run(
        image_path=image_path,
        caption=caption,
    )
    result.print_summary()
    results.append((img_id, result))

# Summarize
n_corrected = sum(1 for _, r in results if r.corrected_caption != r.original_caption)
n_hallucinated = sum(1 for _, r in results if r.any_hallucination)
avg_edit = sum(r.edit_rate for _, r in results) / max(len(results), 1)

print(f'\n{"="*60}')
print(f'  FAST TEST SUMMARY ({len(results)} images)')
print(f'{"="*60}')
print(f'  Hallucinations detected: {n_hallucinated}/{len(results)}')
print(f'  Captions corrected:      {n_corrected}/{len(results)}')
print(f'  Avg edit rate:           {avg_edit:.4f}')
print(f'{"="*60}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 5 — B3 Baseline (LLM Direct Correction) on same subset
# ═══════════════════════════════════════════════════════════════
import together
import time

client = together.Together(api_key=os.environ.get('TOGETHER_API_KEY'))
B3_MODEL = 'meta-llama/Llama-3.3-70B-Instruct-Turbo'

B3_SYSTEM = """You are a precise image caption editor. Review the given caption and fix \
any potentially incorrect relationships, spatial descriptions, or actions described. \
Make minimal changes — only fix things that seem factually unlikely or inconsistent. \
If the caption seems fine, return it unchanged. Output ONLY the corrected caption."""

B3_USER = 'Caption: "{caption}"\n\nReview and fix any incorrect relationships. Output ONLY the corrected caption:'

b3_results = {}  # image_id → corrected caption

for img_id in sorted(id_to_caption.keys()):
    if img_id not in id_to_path:
        continue
    caption = id_to_caption[img_id]
    
    for attempt in range(3):
        try:
            resp = client.chat.completions.create(
                model=B3_MODEL,
                messages=[
                    {'role': 'system', 'content': B3_SYSTEM},
                    {'role': 'user', 'content': B3_USER.format(caption=caption)},
                ],
                max_tokens=200,
                temperature=0.2,
            )
            corrected = resp.choices[0].message.content.strip().strip('"').strip("'").strip()
            b3_results[img_id] = corrected
            break
        except Exception as e:
            if attempt < 2:
                time.sleep(2 ** attempt)
            else:
                b3_results[img_id] = caption  # fallback
                print(f'  [B3] Failed for {img_id}: {e}')

b3_changed = sum(1 for img_id, cap in b3_results.items() if cap != id_to_caption.get(img_id, ''))
print(f'\nB3 baseline: {b3_changed}/{len(b3_results)} captions changed')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 6 — Mention-Filtered LLM-Judge Evaluation
# ═══════════════════════════════════════════════════════════════
import re
import time
import together
from tqdm import tqdm

client_judge = together.Together(api_key=os.environ.get('TOGETHER_API_KEY'))
LLM_JUDGE_MODEL = 'meta-llama/Llama-3.3-70B-Instruct-Turbo'

LLM_JUDGE_SYS = """You are a factual judge. Given an image caption and a yes/no question, \
answer based ONLY on what the caption states. Do NOT use any external knowledge.

If the caption explicitly or implicitly supports the answer \"yes\", respond \"yes\".
If the caption contradicts the claim or does not mention it, respond \"no\".

Respond with ONLY \"yes\" or \"no\" — nothing else."""

LLM_JUDGE_USR = 'Caption: \"{caption}\"\n\nQuestion: {question}\n\nBased solely on what the caption says, answer yes or no:'

def llm_judge(caption, question, max_retries=3):
    for attempt in range(max_retries):
        try:
            resp = client_judge.chat.completions.create(
                model=LLM_JUDGE_MODEL,
                messages=[
                    {'role': 'system', 'content': LLM_JUDGE_SYS},
                    {'role': 'user', 'content': LLM_JUDGE_USR.format(
                        caption=caption, question=question)},
                ],
                max_tokens=5, temperature=0.0,
            )
            ans = resp.choices[0].message.content.strip().lower()
            return 'yes' if 'yes' in ans else 'no'
        except Exception:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
            else:
                return 'no'

def compute_rpope_metrics(preds):
    tp = sum(1 for p in preds if p['pred'] == 'yes' and p['gt'] == 'yes')
    tn = sum(1 for p in preds if p['pred'] == 'no'  and p['gt'] == 'no')
    fp = sum(1 for p in preds if p['pred'] == 'yes' and p['gt'] == 'no')
    fn = sum(1 for p in preds if p['pred'] == 'no'  and p['gt'] == 'yes')
    total = tp + tn + fp + fn
    accuracy  = (tp + tn) / total if total else 0
    precision = tp / (tp + fp) if (tp + fp) else 0
    recall    = tp / (tp + fn) if (tp + fn) else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    yes_ratio = (tp + fp) / total if total else 0
    return {'accuracy': round(accuracy, 4), 'precision': round(precision, 4),
            'recall': round(recall, 4), 'f1': round(f1, 4),
            'yes_ratio': round(yes_ratio, 4), 'total': total,
            'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn}

def caption_mentions_relation(caption, question):
    caption_lower = caption.lower()
    q_clean = question.lower()
    for prefix in ['is there ', 'is the ', 'are the ', 'does the ', 'is a ', 'are there ']:
        q_clean = q_clean.replace(prefix, '')
    q_clean = q_clean.rstrip('?').strip()
    stop_words = {'a','an','the','in','on','of','to','and','is','are','it','its',
                  'this','that','with','for','at','by','from','or','be','was',
                  'were','been','being','image','photo','picture','there',
                  'does','do','has','have','can','could'}
    q_words = [w for w in re.findall(r'\\w+', q_clean) if w not in stop_words and len(w) > 2]
    if not q_words:
        return False
    matches = sum(1 for w in q_words if w in caption_lower)
    return (matches / len(q_words)) >= 0.5

# --- Build the evaluation dataframe ---
# Filter R-Bench questions to our fast test images
fast_test_ids = set(img_id for img_id, _ in results)
df_b1_fast = df_b1[df_b1['image_id'].isin(fast_test_ids)].copy()

# Build caption maps
rc_cap_map = {img_id: r.corrected_caption for img_id, r in results}
b1_cap_map = id_to_caption
b3_cap_map = b3_results

# Filter to mention-relevant questions
mentioned_rows = []
for _, row in df_b1_fast.iterrows():
    q = str(row['question'])
    img_id = row['image_id']
    b1_cap = b1_cap_map.get(img_id, '')
    b3_cap = b3_cap_map.get(img_id, '')
    rc_cap = rc_cap_map.get(img_id, '')
    if (caption_mentions_relation(b1_cap, q) or
        caption_mentions_relation(b3_cap, q) or
        caption_mentions_relation(rc_cap, q)):
        mentioned_rows.append(row)

df_eval = pd.DataFrame(mentioned_rows)
print(f'Total R-Bench questions for {len(fast_test_ids)} images: {len(df_b1_fast)}')
print(f'Mention-filtered questions: {len(df_eval)}')

if len(df_eval) < 5:
    print('\n⚠️ Too few mention-filtered questions. Try more images.')
else:
    # Run LLM judge on all three conditions
    judge_results = {'b1': [], 'b3': [], 'rc': []}
    
    for _, row in tqdm(df_eval.iterrows(), total=len(df_eval), desc='LLM Judge'):
        q = str(row['question'])
        gt = str(row['gt']).lower().strip()
        img_id = row['image_id']
        
        b1_pred = llm_judge(b1_cap_map.get(img_id, ''), q)
        b3_pred = llm_judge(b3_cap_map.get(img_id, ''), q)
        rc_pred = llm_judge(rc_cap_map.get(img_id, ''), q)
        
        judge_results['b1'].append({'pred': b1_pred, 'gt': gt})
        judge_results['b3'].append({'pred': b3_pred, 'gt': gt})
        judge_results['rc'].append({'pred': rc_pred, 'gt': gt})
    
    b1_m = compute_rpope_metrics(judge_results['b1'])
    b3_m = compute_rpope_metrics(judge_results['b3'])
    rc_m = compute_rpope_metrics(judge_results['rc'])
    
    print(f'\n{"="*75}')
    print(f'  ★ MENTION-FILTERED LLM-Judge R-POPE (Fast Test) ★')
    print(f'  Questions: {len(df_eval)}')
    print(f'{"="*75}')
    print(f'  {"Metric":<14} {"B1 (Original)":>14} {"B3 (LLM Direct)":>16} {"RelCheck":>14}')
    print(f'  {"-"*58}')
    for key in ['accuracy', 'precision', 'recall', 'f1', 'yes_ratio']:
        print(f'  {key:<14} {b1_m[key]*100:>13.2f}% {b3_m[key]*100:>15.2f}% {rc_m[key]*100:>13.2f}%')
    print(f'\n  Deltas:')
    print(f'    RelCheck vs B1:         {(rc_m["accuracy"] - b1_m["accuracy"])*100:+.2f}%')
    print(f'    RelCheck vs B3:         {(rc_m["accuracy"] - b3_m["accuracy"])*100:+.2f}%')
    print(f'    B3 vs B1:               {(b3_m["accuracy"] - b1_m["accuracy"])*100:+.2f}%')
    print(f'{"="*75}')

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 7 — Qualitative Comparison (inspect corrections)
# ═══════════════════════════════════════════════════════════════
print(f'\n{"="*75}')
print('  QUALITATIVE: Corrected Captions Comparison')
print(f'{"="*75}\n')

for img_id, result in results:
    if result.corrected_caption != result.original_caption:
        b3_cap = b3_results.get(img_id, result.original_caption)
        print(f'Image: {img_id}')
        print(f'  Original:  {result.original_caption}')
        print(f'  B3:        {b3_cap}')
        print(f'  RelCheck:  {result.corrected_caption}')
        print(f'  Edit rate: {result.edit_rate:.4f}')
        # Show evidence
        for t in result.triples:
            if t.hallucinated is True:
                evidence = t.vqa_evidence or 'N/A'
                print(f'  Hallucinated: ({t.subject}, {t.relation}, {t.obj}) → Evidence: "{evidence}"')
        print()

In [ ]:
# ═══════════════════════════════════════════════════════════════
# CELL 8 — Save Results
# ═══════════════════════════════════════════════════════════════
import json

fast_test_output = {
    'n_images': len(results),
    'n_corrected': n_corrected,
    'n_hallucinated': n_hallucinated,
    'avg_edit_rate': round(avg_edit, 4),
    'mention_filtered_questions': len(df_eval) if 'df_eval' in dir() else 0,
    'metrics': {
        'B1': b1_m if 'b1_m' in dir() else {},
        'B3': b3_m if 'b3_m' in dir() else {},
        'RelCheck': rc_m if 'rc_m' in dir() else {},
    },
    'per_image': [
        {
            'image_id': img_id,
            'original': r.original_caption,
            'corrected': r.corrected_caption,
            'b3': b3_results.get(img_id, ''),
            'n_triples': r.n_triples,
            'n_hallucinated': r.n_hallucinated,
            'edit_rate': round(r.edit_rate, 4),
            'evidence': [
                {'triple': f'({t.subject}, {t.relation}, {t.obj})',
                 'evidence': t.vqa_evidence or ''}
                for t in r.triples if t.hallucinated is True
            ],
        }
        for img_id, r in results
    ],
}

output_path = f'{EVAL_DIR}/fast_test_improved.json'
with open(output_path, 'w') as f:
    json.dump(fast_test_output, f, indent=2)
print(f'✅ Results saved to {output_path}')

# Copy to Drive if available
DRIVE_EVAL_DIR = '/content/drive/MyDrive/CS298/eval'
if os.path.exists(DRIVE_EVAL_DIR):
    import shutil
    shutil.copy2(output_path, DRIVE_EVAL_DIR)
    print(f'💾 Synced to Drive')